In [ ]:
import polars as pl
from google.colab import files

NULL_VALUES = ["\\N", r"\N", "", " "]


def limpiar_aeropuertos(ruta_entrada: str) -> pl.DataFrame:
   
    aeropuertos = pl.read_csv(
        ruta_entrada,
        has_header=False,
        new_columns=[
            "id", "name", "city", "country", "iata", "icao",
            "latitude", "longitude", "altitude", "timezone", "dst", "tz", "type", "source"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    filas_originales = aeropuertos.height

    aeropuertos_limpios = aeropuertos.select([
        pl.col("id").cast(pl.Int64, strict=False),
        pl.col("name").str.strip_chars(),
        pl.col("city").str.strip_chars(),
        pl.col("country").str.strip_chars(),
        pl.col("iata").str.strip_chars(),
        pl.col("icao").str.strip_chars(),
        pl.col("latitude").cast(pl.Float64, strict=False),
        pl.col("longitude").cast(pl.Float64, strict=False)
    ]).drop_nulls(["id", "name", "country", "latitude", "longitude"])

    filas_sin_nulos = aeropuertos_limpios.height

    aeropuertos_finales = aeropuertos_limpios.filter(
        (pl.col("latitude") >= -90) & (pl.col("latitude") <= 90) &
        (pl.col("longitude") >= -180) & (pl.col("longitude") <= 180)
    ).unique(subset=["id"], maintain_order=True)

    print(f"Filas originales: {filas_originales}")
    print(f"Eliminados por nulos o coordenadas inválidas: {filas_originales - aeropuertos_finales.height}")
    print(f"Total de vértices/aeropuertos listos: {aeropuertos_finales.height}\n")
    
    return aeropuertos_finales


def procesar_camino_retos_1_4(rutas_raw: pl.DataFrame, aeropuertos_df: pl.DataFrame) -> pl.DataFrame:
   
    rutas_grafo = rutas_raw.select([
        pl.col("source_id").cast(pl.Int64, strict=False),
        pl.col("destination_id").cast(pl.Int64, strict=False)
    ]).drop_nulls(["source_id", "destination_id"])

    rutas_sin_ciclos = rutas_grafo.filter(pl.col("source_id") != pl.col("destination_id"))

    
    rutas_unicas = rutas_sin_ciclos.unique(
        subset=["source_id", "destination_id"],
        maintain_order=True
    )

    ids_validos = aeropuertos_df.select("id")
    rutas_validadas = (
        rutas_unicas
        .join(ids_validos.rename({"id": "source_id"}), on="source_id", how="inner")
        .join(ids_validos.rename({"id": "destination_id"}), on="destination_id", how="inner")
    )

    print(f"Rutas unicas simplificadas encontradas: {rutas_unicas.height}")
    print(f"Rutas descartadas por aeropuertos inexistentes: {rutas_unicas.height - rutas_validadas.height}")
    print(f"Aristas finales para Grafo Simple (Retos 1-4): {rutas_validadas.height}\n")
    
    return rutas_validadas


def procesar_camino_reto_5(rutas_raw: pl.DataFrame, aeropuertos_df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
  
    aerolineas_maestras = rutas_raw.select([
        pl.col("airline_id").cast(pl.Int64, strict=False),
        pl.col("airline_code").str.strip_chars()
    ]).drop_nulls(["airline_id", "airline_code"]).unique(subset=["airline_id"], maintain_order=True)

    print(f"Catálogo de aerolíneas operativas derivado con éxito: {aerolineas_maestras.height} operadoras únicas.")

    rutas_multigrafo = rutas_raw.select([
        pl.col("source_id").cast(pl.Int64, strict=False),
        pl.col("destination_id").cast(pl.Int64, strict=False),
        pl.col("airline_id").cast(pl.Int64, strict=False)
    ]).drop_nulls(["source_id", "destination_id", "airline_id"])

    rutas_m_sin_ciclos = rutas_multigrafo.filter(pl.col("source_id") != pl.col("destination_id"))

    rutas_m_unicas = rutas_m_sin_ciclos.unique(
        subset=["source_id", "destination_id", "airline_id"],
        maintain_order=True
    )

    ids_aeropuertos = aeropuertos_df.select("id")
    ids_aerolineas = aerolineas_maestras.select("airline_id")

    rutas_m_validadas = (
        rutas_m_unicas
        .join(ids_aeropuertos.rename({"id": "source_id"}), on="source_id", how="inner")
        .join(ids_aeropuertos.rename({"id": "destination_id"}), on="destination_id", how="inner")
        .join(ids_aerolineas, on="airline_id", how="inner")
    )

    print(f"Rutas competitivas (paralelas) pre-calculadas: {rutas_m_unicas.height}")
    print(f"Aristas finales puras para Multígrafo (Reto 5): {rutas_m_validadas.height}\n")

    return aerolineas_maestras, rutas_m_validadas


def ejecutar_pipeline_global_unificado() -> None:


   
    aeropuertos_limpios_df = limpiar_aeropuertos("airports.dat")

    rutas_raw = pl.read_csv(
        "routes.dat",
        has_header=False,
        new_columns=[
            "airline_code", "airline_id", "source_iata", "source_id",
            "destination_iata", "destination_id", "codeshare", "stops", "equipment"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    rutas_simples_finales = procesar_camino_retos_1_4(rutas_raw, aeropuertos_limpios_df)

    aerolineas_maestras_df, rutas_multigrafo_finales = procesar_camino_reto_5(rutas_raw, aeropuertos_limpios_df)

    aeropuertos_limpios_df.write_csv("airports_clean_final.csv")
    rutas_simples_finales.write_csv("routes_graph_validated.csv")
    aerolineas_maestras_df.write_csv("airlines_clean_final.csv")
    rutas_multigrafo_finales.write_csv("routes_multigraph_v5.csv")

   
    print(" -> [Retos 1 al 5]  airports_clean_final.csv  (Catalogo Unico de Nodos)")
    print(" -> [Retos 1 al 3]  routes_graph_validated.csv (Aristas Simples: Origen, Destino)")
    print(" -> [Solo Reto 5]   airlines_clean_final.csv  (Catalogo de Operadoras Generadas)")
    print(" -> [Solo Reto 5]   routes_multigraph_v5.csv  (Aristas de 3 Columnas para Multigrafo)")


ejecutar_pipeline_global_unificado()

=== REPORTE DE LIMPIEZA DE RUTAS ===
Filas originales: 67663
Filas eliminadas por IDs faltantes: 423
Self-loops (origen == destino) eliminados: 1
Duplicados eliminados (ID Origen + ID Destino): 29966
Duplicados eliminados (Aerolínea + IDs): 0
------------------------------------
Filas finales en routes_graph_clean.csv: 37273
Filas finales en routes_airline_clean.csv: 67239
Filas finales en routes_full_clean.csv: 67239
------------------------------------
Aerolíneas únicas: 568
Aeropuertos origen únicos: 3315
Aeropuertos destino únicos: 3321
